In [1]:
! which python

'which' is not recognized as an internal or external command,
operable program or batch file.


# Homework 5

- All questions must be answered in your own words, do not copy-paste text from the internet. Points can be deducted for terrible formatting or incomprehensible English.

- Code must be commented. If you use code you found online, you have to add the link to the source you used. There is no penalty for using outside sources as long as you convince us you understand the code.

**To pass the homework you need to complete 100% of the homework.**

**Once completed zip the entire directory containing this exercise and upload it to Moodle.**

## Fully-Connected Neural Nets

In the previous homework you implemented a fully-connected two-layer neural network on CIFAR-10. The implementation was simple but not very modular since the loss and gradient were computed in a single monolithic function. This is manageable for a simple two-layer network, but would become impractical as we move to bigger models. Ideally we want to build networks using a more modular design so that we can implement different layer types in isolation and then snap them together into models with different architectures.

In this exercise we will implement fully-connected networks using a more modular approach. For each layer we will implement a `forward` and a `backward` function. The `forward` function will receive inputs, weights, and other parameters and will return both an output and a `cache` object storing data needed for the backward pass, like this:

```python
def layer_forward(x, w):
  """ Receive inputs x and weights w """
  # Do some computations ...
  z = # ... some intermediate value
  # Do some more computations ...
  out = # the output
   
  cache = (x, w, z, out) # Values we need to compute gradients
   
  return out, cache
```

The backward pass will receive upstream derivatives and the `cache` object, and will return gradients with respect to the inputs and weights, like this:

```python
def layer_backward(dout, cache):
  """
  Receive derivative of loss with respect to outputs and cache,
  and compute derivative with respect to inputs.
  """
  # Unpack cache values
  x, w, z, out = cache
  
  # Use values in cache to compute derivatives
  dx = # Derivative of loss with respect to x
  dw = # Derivative of loss with respect to w
  
  return dx, dw
```

After implementing a bunch of layers this way, we will be able to easily combine them to build classifiers with different architectures.

For additional background reading see http://cs231n.github.io/neural-networks-1/, http://cs231n.github.io/neural-networks-2/ and http://cs231n.github.io/neural-networks-3/.


In [2]:
import torch

from fc_net import *
from data_utils import get_CIFAR10_data
from gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from solver import Solver

import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

def rel_error(x, y):

    """ returns relative error """

    x = x.to(torch.float64)
    y = y.to(torch.float64)
    
    return torch.max(torch.abs(x - y) / (torch.max(torch.tensor(1e-8, dtype=torch.float64), torch.abs(x) + torch.abs(y))))

In [5]:
# set default tensor type
torch.set_default_dtype(torch.float64)

# Load the (preprocessed) CIFAR10 data.
data = get_CIFAR10_data('datasets/cifar-10-batches-py/')

for k, v in list(data.items()):
    
    data[k] = torch.tensor(v, dtype=torch.float64)
    print('%s: ' % k, data[k].shape)

d:\Artem\Tartu_CS\1st year\Neural Nets\HW5\data_utils.py:15: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return  pickle.load(f, encoding='latin1')


X_train:  torch.Size([49000, 3, 32, 32])
y_train:  torch.Size([49000])
X_val:  torch.Size([1000, 3, 32, 32])
y_val:  torch.Size([1000])
X_test:  torch.Size([1000, 3, 32, 32])
y_test:  torch.Size([1000])


## Affine layer: foward

**Task 5.1.1**

Open the file `layers.py` and implement the `affine_forward` function. Once you are done you can test your implementaion by running the following:

In [12]:

num_inputs = 2
input_shape = (4, 5, 6)
output_dim = 3

input_size = num_inputs * int(torch.prod(torch.tensor(input_shape)).item())
weight_size = output_dim * int(torch.prod(torch.tensor(input_shape)).item())

x = torch.linspace(-0.1, 0.5, steps=input_size).reshape(num_inputs, *input_shape)
w = torch.linspace(-0.2, 0.3, steps=weight_size).reshape(int(torch.prod(torch.tensor(input_shape)).item()), output_dim)
b = torch.linspace(-0.3, 0.1, steps=output_dim)

In [11]:
b.shape

torch.Size([3])

In [13]:
# Test the affine_forward function
out, _ = affine_forward(x, w, b)

correct_out = torch.tensor([[ 1.49834967,  1.70660132,  1.91485297],
                             [ 3.25553199,  3.5141327,   3.77273342]])

# Compare your output with ours. The error should be around 1e-9.
print('Testing `affine_forward` function:')
print(f'out:\n{out}')

diff_ = rel_error(out, correct_out)

print()
assert diff_ < 1e-9, f'Task 5.1.1 failed: difference too large ({diff_:.6e})'

print('Task 5.1.1 passed error is small: ', diff_)

Testing `affine_forward` function:
out:
tensor([[1.4983, 1.7066, 1.9149],
        [3.2555, 3.5141, 3.7727]])

Task 5.1.1 passed error is small:  tensor(9.7698e-10)


## Affine layer: backward

**Task 5.1.2**

Now implement the `affine_backward` function and test your implementation using numeric gradient checking.

In [23]:
# Test the affine_backward function
torch.manual_seed(231)

x = torch.randn(10, 2, 3, dtype=torch.float64)
w = torch.randn(6, 5, dtype=torch.float64)
b = torch.randn(5, dtype=torch.float64)
dout = torch.randn(10, 5, dtype=torch.float64)

dx_num = eval_numerical_gradient_array(lambda x: affine_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_forward(x, w, b)[0], b, dout)

_, cache = affine_forward(x, w, b)
dx, dw, db = affine_backward(dout, cache)

dx_error = rel_error(dx_num, dx)
dw_error = rel_error(dw_num, dw)
db_error = rel_error(db_num, db)

print()
assert dx_error < 1e-9, f'Task 5.1.2 failed: dx error too large ({dx_error:.6e})'
assert dw_error < 1e-9, f'Task 5.1.2 failed: dw error too large ({dw_error:.6e})'
assert db_error < 1e-9, f'Task 5.1.2 failed: db error too large ({db_error:.6e})'

# The error should now be around 1e-9
print('Task 5.1.2 passed errors are is small: ')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))


Task 5.1.2 passed errors are is small: 
dx error:  tensor(8.6668e-11)
dw error:  tensor(4.0141e-11)
db error:  tensor(1.4949e-11)


## ReLU layer: forward

**Taks 5.1.3** 

Implement the forward pass for the ReLU activation function in the `relu_forward` function and test your implementation using the following:

In [24]:
# Test the relu_forward function

x = torch.linspace(-0.5, 0.5, steps=12).reshape(3, 4)

out, _ = relu_forward(x)
correct_out = torch.tensor([[ 0.,          0.,          0.,          0.,        ],
                             [ 0.,          0.,          0.04545455,  0.13636364,],
                             [ 0.22727273,  0.31818182,  0.40909091,  0.5,       ]], dtype=torch.float64)

# Compare your output with ours. The error should be around 5e-8
print(f'out:\n{out}')
print('Testing relu_forward function:')

diff_ = rel_error(out, correct_out)

print()
assert diff_ < 5e-8, f'Task 5.1.3 failed: difference too large ({diff_:.6e})'

print('Task 5.1.3 passed difference is small: ', diff_)

out:
tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0455, 0.1364],
        [0.2273, 0.3182, 0.4091, 0.5000]])
Testing relu_forward function:

Task 5.1.3 passed difference is small:  tensor(5.0000e-08)


## ReLU layer: backward

**Task 5.1.4** 

Now implement the backward pass for the ReLU activation function in the `relu_backward` function and test your implementation using numeric gradient checking:

In [28]:
torch.manual_seed(231)

x = torch.randn(10, 10)
dout = torch.randn_like(x)

dx_num = eval_numerical_gradient_array(lambda x: relu_forward(x)[0], x, dout)

_, cache = relu_forward(x)
dx = relu_backward(dout, cache)

dx_error = rel_error(dx_num, dx)

print()
assert dx_error < 1e-11, f'Task 5.1.4 failed: dx error too large ({dx_error:.6e})'

# The error should be around 1e-11
print('Task 5.1.4 passed difference is small:')
print('dx error: ', rel_error(dx_num, dx))


Task 5.1.4 passed difference is small:
dx error:  tensor(3.2756e-12)


## "Sandwich" layers
There are some common patterns of layers that are frequently used in neural nets. For example, affine layers are frequently followed by a ReLU nonlinearity. To make these common patterns easy, we define several convenience layers in the file `layer_utils.py`.

For now take a look at the `affine_relu_forward` and `affine_relu_backward` functions, and run the following to numerically gradient check the backward pass:

In [29]:
from layer_utils import affine_relu_forward, affine_relu_backward

torch.manual_seed(231)

x = torch.randn(2, 3, 4)
w = torch.randn(12, 10)
b = torch.randn(10)
dout = torch.randn(2, 10)

out, cache = affine_relu_forward(x, w, b)
dx, dw, db = affine_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: affine_relu_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_relu_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_relu_forward(x, w, b)[0], b, dout)

dx_error = rel_error(dx_num, dx)
dw_error = rel_error(dw_num, dw)
db_error = rel_error(db_num, db)

print()
assert dx_error < 1e-8, f'Task failed: dx error too large ({dx_error:.6e})'
assert dw_error < 1e-9, f'Task failed: dw error too large ({dw_error:.6e})'
assert db_error < 1e-9, f'Task failed: db error too large ({db_error:.6e})'

print('Testing:')
print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))



Testing:
dx error:  tensor(1.1069e-09)
dw error:  tensor(3.1158e-10)
db error:  tensor(2.9626e-11)


## Softmax loss layer

You implemented this loss function in the last assignment, so we'll give it to you for free here. You should still make sure you understand how they work by looking at the implementation in `layers.py`.

You can make sure that the implementations are correct by running the following:

In [30]:
torch.manual_seed(231)

num_classes, num_inputs = 10, 50

x = 0.001 * torch.randn(num_inputs, num_classes)
y = torch.randint(num_classes, (num_inputs,))

dx_num = eval_numerical_gradient(lambda x: softmax_loss(x, y)[0], x, verbose=False)
loss, dx = softmax_loss(x, y)

print()
assert abs( round(loss.item(), 1) - 2.3) < 1e-10, f'Loss should be around 2.3 but got {loss:.6e}'
assert rel_error(dx_num, dx) < 1e-8, f'dx error too large ({rel_error(dx_num, dx):.6e})'

# Test softmax_loss function. Loss should be 2.3 and dx error should be 1e-8
print('Testing softmax_loss:')
print('loss: ', loss)
print('dx error: ', rel_error(dx_num, dx))


Testing softmax_loss:
loss:  tensor(2.3027)
dx error:  tensor(9.2222e-09)


## Two-layer network

In the previous assignment you implemented a two-layer neural network in a single monolithic class. Now that you have implemented modular versions of the necessary layers, you will reimplement the two layer network using these modular implementations.

**Task 5.2** 

Open the file `fc_net.py` and complete the implementation of the `TwoLayerNet` class. This class will serve as a model for the other networks you will implement in this assignment, so read through it to make sure you understand the API. You can run the cell below to test your implementation.

In [48]:
torch.manual_seed(231)

N, D, H, C = 3, 5, 50, 7
X = torch.randn(N, D)
y = torch.randint(C, (N,))

std = 1e-3
model = TwoLayerNet(input_dim=D, hidden_dim=H, num_classes=C, weight_scale=std)

print('Testing initialization ... ')

W1_std = abs(model.params['W1'].std() - std)
b1 = model.params['b1']
W2_std = abs(model.params['W2'].std() - std)
b2 = model.params['b2']

assert W1_std < std / 10, 'First layer weights do not seem right'
assert torch.all(b1 == 0), 'First layer biases do not seem right'
assert W2_std < std / 10, 'Second layer weights do not seem right'
assert torch.all(b2 == 0), 'Second layer biases do not seem right'

print('Testing test-time forward pass ... ')
model.params['W1'] = torch.linspace(-0.7, 0.3, steps=D * H).reshape(D, H)
model.params['b1'] = torch.linspace(-0.1, 0.9, steps=H)
model.params['W2'] = torch.linspace(-0.3, 0.4, steps=H * C).reshape(H, C)
model.params['b2'] = torch.linspace(-0.9, 0.1, steps=C)

X = torch.linspace(-5.5, 4.5, steps=N * D).reshape(D, N).T

scores = model.loss(X = X)

correct_scores = torch.tensor(
  [[11.53165108,  12.2917344,   13.05181771,  13.81190102,  14.57198434, 15.33206765,  16.09215096],
   [12.05769098,  12.74614105,  13.43459113,  14.1230412,   14.81149128, 15.49994135,  16.18839143],
   [12.58373087,  13.20054771,  13.81736455,  14.43418138,  15.05099822, 15.66781506,  16.2846319 ]])

scores_diff = torch.abs(scores - correct_scores).sum()

assert scores_diff < 1e-6, 'Problem with test-time forward pass'

print('Testing training loss (no regularization)')
y = torch.tensor([0, 5, 1], dtype=torch.long)

loss, grads = model.loss(X, y)
correct_loss = 3.4702243556

assert abs(loss - correct_loss) < 1e-10, 'Problem with training-time loss'

model.reg = 1.0
loss, grads = model.loss(X, y)
correct_loss = 26.5948426952

assert abs(loss - correct_loss) < 1e-10, f'Problem with regularization loss, {loss}'

for reg in [0.0, 0.7]:

    print('Running numeric gradient check with reg = ', reg)

    model.reg = reg
    loss, grads = model.loss(X, y)

    for name in sorted(grads):

        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False)
        
        print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))
        
        assert rel_error(grad_num, grads[name]) < 0.6, "Error with gradient for " + name

print('Task 5.2 passed!')

Testing initialization ... 
Testing test-time forward pass ... 
Testing training loss (no regularization)
Running numeric gradient check with reg =  0.0
W1 relative error: 1.52e-08
W2 relative error: 5.63e-10
b1 relative error: 1.35e-08
b2 relative error: 2.53e-10
Running numeric gradient check with reg =  0.7
W1 relative error: 3.12e-07
W2 relative error: 7.98e-08
b1 relative error: 1.56e-08
b2 relative error: 1.97e-09
Task 5.2 passed!


## Solver
In the previous assignment, the logic for training models was coupled to the models themselves. Following a more modular design, for this assignment we have split the logic for training models into a separate class.

**Task 5.3** 

Open the file `solver.py` and read through it to familiarize yourself with the API. After doing so, use a `Solver` instance to train a `TwoLayerNet` that achieves at least `50%` accuracy on the validation set.

In [ ]:
model = TwoLayerNet()
solver = None

##############################################################################
# Task 5.3                                                                   #
# TODO: Use a Solver instance to train a TwoLayerNet that achieves at least  #
# 50% accuracy on the validation set.                                        #
##############################################################################

##############################################################################
#                             END OF YOUR CODE                               #
##############################################################################

print('Task 5.3 passed!')

In [ ]:
# Run this cell to visualize training loss and train / val accuracy

plt.subplot(2, 1, 1)
plt.title('Training loss')
plt.plot(solver.loss_history, 'o')
plt.xlabel('Iteration')

plt.subplot(2, 1, 2)
plt.title('Accuracy')
plt.plot(solver.train_acc_history, '-o', label='train')
plt.plot(solver.val_acc_history, '-o', label='val')
plt.plot([0.5] * len(solver.val_acc_history), 'k--')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.gcf().set_size_inches(15, 12)
plt.show()

## Multilayer network
Next you will implement a fully-connected network with an arbitrary number of hidden layers. Read through the `FullyConnectedNet` class in the file `fc_net.py`.

**Task 5.4.1** 

Implement the initialization, the forward pass, and the backward pass. For the moment don't worry about implementing dropout or batch normalization; we will add those features soon.

### Initial loss and gradient check

As a sanity check, run the following to check the initial loss and to gradient check the network both with and without regularization. Do the initial losses seem reasonable?

For gradient checking, you should expect to see errors around 1e-6 or less.

In [ ]:
torch.manual_seed(231)

N, D, H1, H2, C = 2, 15, 20, 30, 10
X = torch.randn(N, D)
y = torch.randint(C, (N,))

for reg in [0, 3.14]:

    print('Running check with reg = ', reg)
    model = FullyConnectedNet([H1, H2], input_dim=D, num_classes=C,
                              reg=reg, weight_scale=5e-2, dtype=torch.float64)

    loss, grads = model.loss(X, y)
    print('Initial loss: ', loss)

    for name in sorted(grads):

        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)

        print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))
        
        assert rel_error(grad_num, grads[name]) < 1e-4

print('Task 5.4.1 passed!')

**Task 5.4.2** 

As another sanity check, make sure you can overfit a small dataset of 50 images. First we will try a three-layer network with 100 units in each hidden layer. You will need to tweak the learning rate and initialization scale, but you should be able to overfit and achieve 100% training accuracy within 20 epochs.

In [ ]:
num_train = 50

small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

weight_scale = 1e-2
learning_rate = 1e-2

model = FullyConnectedNet([100, 100],
              weight_scale=weight_scale, dtype=torch.float64)

solver = Solver(model, small_data,
                print_every=10, num_epochs=20, batch_size=25,
                update_rule='sgd',
                optim_config={
                    'learning_rate': learning_rate,
                }
         )

solver.train()

plt.plot(solver.loss_history, 'o')
plt.title('Training loss history')
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.show()

print('Task 5.4.2 passed!')

**Task 5.4.3** 

Now try to use a five-layer network with 100 units on each layer to overfit 50 training examples. Again you will have to adjust the learning rate and weight initialization, but you should be able to achieve 100% training accuracy within 20 epochs.

In [ ]:
num_train = 50

small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

learning_rate = 1e-3
weight_scale = 1e-1

model = FullyConnectedNet([100, 100, 100, 100],
                weight_scale=weight_scale, dtype=torch.float64)

solver = Solver(model, small_data,
                print_every=10, num_epochs=20, batch_size=25,
                update_rule='sgd',
                optim_config={
                    'learning_rate': learning_rate,
                }
         )

solver.train()

plt.plot(solver.loss_history, 'o')
plt.title('Training loss history')
plt.xlabel('Iteration')
plt.ylabel('Training loss')
plt.show()

print('Task 5.4.3 passed!')

**Task 5.4.4** 

Did you notice anything about the comparative difficulty of training the three-layer net vs training the five layer net?

**Your Answer:** fill this in

# Update rules
So far we have used vanilla stochastic gradient descent (SGD) as our update rule. More sophisticated update rules can make it easier to train deep networks. We will implement a few of the most commonly used update rules and compare them to vanilla SGD.

## SGD + Momentum
Stochastic gradient descent with momentum is a widely used update rule that tends to make deep networks converge faster than vanilla stochastic gradient descent.

**Task 5.4.5** 

Open the file `optim.py` and read the documentation at the top of the file to make sure you understand the API. Implement the ( SGD + momentum ) update rule in the function `sgd_momentum` and run the following to check your implementation. You should see errors less than 1e-8.

In [ ]:
from optim import sgd_momentum

N, D = 4, 5
w = torch.linspace(-0.4, 0.6, steps=N * D).reshape(N, D)
dw = torch.linspace(-0.6, 0.4, steps=N * D).reshape(N, D)
v = torch.linspace(0.6, 0.9, steps=N * D).reshape(N, D)

config = {'learning_rate': 1e-3, 'velocity': v}
next_w, _ = sgd_momentum(w, dw, config=config)

expected_next_w = torch.tensor([
  [ 0.1406,      0.20738947,  0.27417895,  0.34096842,  0.40775789],
  [ 0.47454737,  0.54133684,  0.60812632,  0.67491579,  0.74170526],
  [ 0.80849474,  0.87528421,  0.94207368,  1.00886316,  1.07565263],
  [ 1.14244211,  1.20923158,  1.27602105,  1.34281053,  1.4096    ]])
expected_velocity = torch.tensor([
  [ 0.5406,      0.55475789,  0.56891579, 0.58307368,  0.59723158],
  [ 0.61138947,  0.62554737,  0.63970526,  0.65386316,  0.66802105],
  [ 0.68217895,  0.69633684,  0.71049474,  0.72465263,  0.73881053],
  [ 0.75296842,  0.76712632,  0.78128421,  0.79544211,  0.8096    ]])

print()
assert rel_error(next_w, expected_next_w) < 1e-8, f'Task 5.4.5 failed: next_w error too large ({rel_error(next_w, expected_next_w):.6e})'
assert rel_error(expected_velocity, config['velocity']) < 1e-8, f'Task 5.4.5 failed: velocity error too large ({rel_error(expected_velocity, config["velocity"]):.6e})'

print('Task 5.4.5 passed!')
print('next_w error: ', rel_error(next_w, expected_next_w))
print('velocity error: ', rel_error(expected_velocity, config['velocity']))

Once you have done so, run the following to train a six-layer network with both SGD and SGD+momentum. You should see the ( SGD + momentum ) update rule converge faster.

In [ ]:
num_train = 4000

small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

solvers = {}

for update_rule in ['sgd', 'sgd_momentum']:

    print('running with ', update_rule)
    model = FullyConnectedNet([100, 100, 100, 100, 100], weight_scale=5e-2)

    solver = Solver(model, small_data,
                    num_epochs=5, batch_size=100,
                    update_rule=update_rule,
                    optim_config={'learning_rate': 1e-2,},
                    verbose=True)
    
    solvers[update_rule] = solver
    solver.train()
    print()

plt.subplot(3, 1, 1)
plt.title('Training loss')
plt.xlabel('Iteration')

plt.subplot(3, 1, 2)
plt.title('Training accuracy')
plt.xlabel('Epoch')

plt.subplot(3, 1, 3)
plt.title('Validation accuracy')
plt.xlabel('Epoch')

for update_rule, solver in list(solvers.items()):

    plt.subplot(3, 1, 1)
    plt.plot(solver.loss_history, 'o', label=update_rule)

    plt.subplot(3, 1, 2)
    plt.plot(solver.train_acc_history, '-o', label=update_rule)

    plt.subplot(3, 1, 3)
    plt.plot(solver.val_acc_history, '-o', label=update_rule)

for i in [1, 2, 3]:
    
    plt.subplot(3, 1, i)
    plt.legend(loc='upper center', ncol=4)

plt.gcf().set_size_inches(15, 15)
plt.show()

## RMSProp and Adam
RMSProp [1] and Adam [2] are update rules that set per-parameter learning rates by using a running average of the second moments of gradients.

**Task 5.4.6** 

In the file `optim.py`, implement the RMSProp update rule in the `rmsprop` function and implement the Adam update rule in the `adam` function, and check your implementations using the tests below.

[1] Tijmen Tieleman and Geoffrey Hinton. "Lecture 6.5-rmsprop: Divide the gradient by a running average of its recent magnitude." COURSERA: Neural Networks for Machine Learning 4 (2012).

[2] Diederik Kingma and Jimmy Ba, "Adam: A Method for Stochastic Optimization", ICLR 2015.

In [ ]:
# Test RMSProp implementation; you should see errors less than 1e-7
from optim import rmsprop

N, D = 4, 5
w = torch.linspace(-0.4, 0.6, steps=N * D).reshape(N, D)
dw = torch.linspace(-0.6, 0.4, steps=N * D).reshape(N, D)
cache = torch.linspace(0.6, 0.9, steps=N * D).reshape(N, D)

config = {'learning_rate': 1e-2, 'cache': cache}
next_w, _ = rmsprop(w, dw, config=config)

expected_next_w = torch.tensor([
  [-0.39223849, -0.34037513, -0.28849239, -0.23659121, -0.18467247],
  [-0.132737,   -0.08078555, -0.02881884,  0.02316247,  0.07515774],
  [ 0.12716641,  0.17918792,  0.23122175,  0.28326742,  0.33532447],
  [ 0.38739248,  0.43947102,  0.49155973,  0.54365823,  0.59576619]])

expected_cache = torch.tensor([
  [ 0.5976,      0.6126277,   0.6277108,   0.64284931,  0.65804321],
  [ 0.67329252,  0.68859723,  0.70395734,  0.71937285,  0.73484377],
  [ 0.75037008,  0.7659518,   0.78158892,  0.79728144,  0.81302936],
  [ 0.82883269,  0.84469141,  0.86060554,  0.87657507,  0.8926    ]])

print()
assert rel_error(next_w, expected_next_w) < 1e-7, f'Task 5.4.6 failed: next_w error too large ({rel_error(next_w, expected_next_w):.6e})'
assert rel_error(expected_cache, config['cache']) < 1e-7, f'Task 5.4.6 failed: cache error too large ({rel_error(expected_cache, config["cache"]):.6e})'

print('Task 5.4.6 passed!')
print('next_w error: ', rel_error(expected_next_w, next_w))
print('cache error: ', rel_error(expected_cache, config['cache']))

In [ ]:
# Test Adam implementation; you should see errors around 1e-7 or less
from optim import adam

N, D = 4, 5
w = torch.linspace(-0.4, 0.6, steps=N * D).reshape(N, D)
dw = torch.linspace(-0.6, 0.4, steps=N * D).reshape(N, D)
m = torch.linspace(0.6, 0.9, steps=N * D).reshape(N, D)
v = torch.linspace(0.7, 0.5, steps=N * D).reshape(N, D)

config = {'learning_rate': 1e-2, 'm': m, 'v': v, 't': 5}
next_w, _ = adam(w, dw, config=config)

expected_next_w = torch.tensor([
  [-0.40094747, -0.34836187, -0.29577703, -0.24319299, -0.19060977],
  [-0.1380274,  -0.08544591, -0.03286534,  0.01971428,  0.0722929],
  [ 0.1248705,   0.17744702,  0.23002243,  0.28259667,  0.33516969],
  [ 0.38774145,  0.44031188,  0.49288093,  0.54544852,  0.59801459]])

expected_v = torch.tensor([
  [ 0.69966,     0.68908382,  0.67851319,  0.66794809,  0.65738853,],
  [ 0.64683452,  0.63628604,  0.6257431,   0.61520571,  0.60467385,],
  [ 0.59414753,  0.58362676,  0.57311152,  0.56260183,  0.55209767,],
  [ 0.54159906,  0.53110598,  0.52061845,  0.51013645,  0.49966,   ]])

expected_m = torch.tensor([
  [ 0.48,        0.49947368,  0.51894737,  0.53842105,  0.55789474],
  [ 0.57736842,  0.59684211,  0.61631579,  0.63578947,  0.65526316],
  [ 0.67473684,  0.69421053,  0.71368421,  0.73315789,  0.75263158],
  [ 0.77210526,  0.79157895,  0.81105263,  0.83052632,  0.85      ]])

print()
assert rel_error(next_w, expected_next_w) < 1e-6, f'Task 5.4.6 failed: next_w error too large ({rel_error(next_w, expected_next_w):.6e})'
assert rel_error(expected_v, config['v']) < 1e-6, f'Task 5.4.6 failed: v error too large ({rel_error(expected_v, config["v"]):.6e})'
assert rel_error(expected_m, config['m']) < 1e-6, f'Task 5.4.6 failed: m error too large ({rel_error(expected_m, config["m"]):.6e})'

print('Task 5.4.6 passed!')
print('next_w error: ', rel_error(expected_next_w, next_w))
print('v error: ', rel_error(expected_v, config['v']))
print('m error: ', rel_error(expected_m, config['m']))

Once you have debugged your RMSProp and Adam implementations, run the following to train a pair of deep networks using these new update rules:

In [ ]:
learning_rates = {'rmsprop': 1e-4, 'adam': 1e-3}

for update_rule in ['adam', 'rmsprop']:

    print('running with ', update_rule)
    model = FullyConnectedNet([100, 100, 100, 100, 100], weight_scale=5e-2)

    solver = Solver(model, small_data,
                    num_epochs=5, batch_size=100,
                    update_rule=update_rule,
                    optim_config={'learning_rate': learning_rates[update_rule]},
                    verbose=True)
    
    solvers[update_rule] = solver
    solver.train()
    print()

plt.subplot(3, 1, 1)
plt.title('Training loss')
plt.xlabel('Iteration')

plt.subplot(3, 1, 2)
plt.title('Training accuracy')
plt.xlabel('Epoch')

plt.subplot(3, 1, 3)
plt.title('Validation accuracy')
plt.xlabel('Epoch')

for update_rule, solver in list(solvers.items()):

    plt.subplot(3, 1, 1)
    plt.plot(solver.loss_history, 'o', label=update_rule)

    plt.subplot(3, 1, 2)
    plt.plot(solver.train_acc_history, '-o', label=update_rule)

    plt.subplot(3, 1, 3)
    plt.plot(solver.val_acc_history, '-o', label=update_rule)

for i in [1, 2, 3]:
    
    plt.subplot(3, 1, i)
    plt.legend(loc='upper center', ncol=4)

plt.gcf().set_size_inches(15, 15)
plt.show()

## Train a good model!

**Task 5.5** 

Train the best fully-connected model that you can on CIFAR-10, storing your best model in the `best_model` variable. We require you to get at least 50% accuracy on the validation set using a fully-connected net.

If you are careful it should be possible to get accuracies above 55%, but we don't require it for this part and won't assign extra credit for doing so. Later in the assignment we will ask you to train the best convolutional network that you can on CIFAR-10, and we would prefer that you spend your effort working on convolutional nets rather than fully-connected nets.

You might find it useful to complete the `Dropout.ipynb` notebook before completing this part, since this technique can help you train powerful models.

In [ ]:
best_model = None
################################################################################
# TODO: Train the best FullyConnectedNet that you can on CIFAR-10. You might   #
# batch normalization and dropout useful. Store your best model in the         #
# best_model variable.                                                         #
################################################################################
pass
################################################################################
#                              END OF YOUR CODE                                #
################################################################################

## Test you model
Run your best model on the validation and test sets. You should achieve above 50% accuracy on the validation set.

In [ ]:
assert test_acc > 0.5, f'Test-set requirement failed: accuracy too low ({test_acc:.4f} <= 0.5)'
print('Test set accuracy: ', (test_acc))

print('Task 5.5 passed!')